<a href="https://colab.research.google.com/github/manekaM/DSGP_Project_Individual_Commits/blob/main/Rectification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# 1. Import Libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 2. Load Dataset
df = pd.read_csv("/content/structured_rectification_dataset.csv")

# 3. Preprocessing


# Fill missing values (safety step)
df['Previous_Rectification_Action'] = df['Previous_Rectification_Action'].fillna("No Action")

# Split Fault_Location into numeric columns
df[['String_No', 'Module_No']] = df['Fault_Location'].str.extract(r'String-(\d+)_Module-(\d+)')
df['String_No'] = df['String_No'].astype(int)
df['Module_No'] = df['Module_No'].astype(int)

# Convert Timestamp into useful features
df['Timestamp'] = pd.to_datetime(df['Timestamp'])
df['Month'] = df['Timestamp'].dt.month
df['Hour'] = df['Timestamp'].dt.hour

# Drop unused columns
df.drop(['Fault_Location', 'Timestamp'], axis=1, inplace=True)


# 4. Define Features and Target
X = df.drop('Rectification_Action', axis=1)
y = df['Rectification_Action']

# 5. Define Categorical & Numerical Columns

categorical_cols = [
    'Fault_Type',
    'Severity_Level',
    'Weather_Condition',
    'Previous_Rectification_Action'
]

numeric_cols = [
    'Module_Age_Years',
    'Irradiance',
    'String_No',
    'Module_No',
    'Month',
    'Hour'
]

# 6. Preprocessing Pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
        ('num', 'passthrough', numeric_cols)
    ]
)

# 7. Decision Tree Model
dt = DecisionTreeClassifier(random_state=42)

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', dt)
])

# 8. Hyperparameter Tuning
param_grid = {
    'classifier__max_depth': [5, 10, 15, None],
    'classifier__min_samples_split': [2, 5, 10],
    'classifier__min_samples_leaf': [1, 2, 4],
    'classifier__criterion': ['gini', 'entropy']
}

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1
)

# 9. Train Model
grid_search.fit(X, y)

print("Best Parameters:", grid_search.best_params_)
print("Best Cross-Validation Accuracy:", round(grid_search.best_score_ * 100, 2), "%")

# 10. Final Train-Test Evaluation
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

best_model = grid_search.best_estimator_
best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_test)

print("\nFinal Test Accuracy:", round(accuracy_score(y_test, y_pred) * 100, 2), "%")
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Best Parameters: {'classifier__criterion': 'gini', 'classifier__max_depth': 10, 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 2}
Best Cross-Validation Accuracy: 100.0 %

Final Test Accuracy: 100.0 %

Classification Report:
                        precision    recall  f1-score   support

          Adjust Tilt       1.00      1.00      1.00        70
          Clean Panel       1.00      1.00      1.00       144
        Inspect Cells       1.00      1.00      1.00        77
 Inspect Junction Box       1.00      1.00      1.00        78
     Reinstall Module       1.00      1.00      1.00        77
   Remove Obstruction       1.00      1.00      1.00        81
    Replace Connector       1.00      1.00      1.00        72
         Replace Fuse       1.00      1.00      1.00        73
       Replace Module       1.00      1.00      1.00       227
Replace Panel Section       1.00      1.00      1.00        74
       Replace Wiring       1.00      1.00      1.00       1

In [6]:
# 11. Manual Prediction Check (Sample)

# Take 10 random samples from test set
sample_data = X_test.sample(10, random_state=42)
sample_actual = y_test.loc[sample_data.index]

# Predict
sample_predicted = best_model.predict(sample_data)

# Create comparison table
comparison_df = pd.DataFrame({
    "Fault_Type": sample_data["Fault_Type"],
    "Severity_Level": sample_data["Severity_Level"],
    "Actual_Action": sample_actual,
    "Predicted_Action": sample_predicted
})

comparison_df["Correct?"] = comparison_df["Actual_Action"] == comparison_df["Predicted_Action"]

print(comparison_df)

         Fault_Type Severity_Level       Actual_Action    Predicted_Action  \
3320        Hotspot            Low         Clean Panel         Clean Panel   
2290      Shadowing         Medium  Remove Obstruction  Remove Obstruction   
5160  Short Circuit       Critical      Replace Module      Replace Module   
4874  Short Circuit         Medium        Replace Fuse        Replace Fuse   
3341        Hotspot       Critical      Replace Module      Replace Module   
2278        Hotspot            Low         Clean Panel         Clean Panel   
5427        Hotspot       Critical      Replace Module      Replace Module   
4721      Shadowing           High         Adjust Tilt         Adjust Tilt   
3539   Open Circuit            Low  Tighten Connection  Tighten Connection   
5498      Shadowing       Critical    Reinstall Module    Reinstall Module   

      Correct?  
3320      True  
2290      True  
5160      True  
4874      True  
3341      True  
2278      True  
5427      True  
4721 

In [7]:
tree = best_model.named_steps['classifier']
print("Number of leaves in the final tree:", tree.get_n_leaves())
print("Tree depth:", tree.get_depth())
print("Best parameters were:", grid_search.best_params_)

Number of leaves in the final tree: 12
Tree depth: 6
Best parameters were: {'classifier__criterion': 'gini', 'classifier__max_depth': 10, 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 2}
